# Land Use Profiling for SP Outlets

For each of the 375 geocoded TOTO outlets, this notebook computes a **land use profile** at three radii (500m, 1000m, 1500m) using official URA Master Plan 2019 zoning data. It then classifies each outlet's neighborhood as residential, commercial or mixed.

## Data sources

| Dataset | Source | Records |
|---------|--------|---------|
| URA Master Plan 2019 Land Use | data.gov.sg | 113,212 zoning polygons |
| HDB Existing Building | data.gov.sg | 13,404 block footprints |
| Planning Area Boundary | data.gov.sg | 55 planning areas |
| Geocoded outlets | `outlets_geocoded.csv` (from `build_dataset.py`) | 375 outlets |

This lets us compute a **residential-commercial ratio** (`rc_ratio`) that directly answers the project question: does an outlet's proximity to residential vs commercial clusters predict winning frequency?

## Output

`outlets_geodata.csv` with 38 columns per outlet, including land use areas at 3 radii, HDB block counts, rc_ratio and neighbourhood classification.

## 1. Setup and Configuration

In [ ]:
import csv
import json
import math
import sys
import time
from collections import defaultdict
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
SUPP_DIR = DATA_DIR / "supplementary"
OUT_DIR = DATA_DIR / "analysis_ready"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GEODATASETS = [
    {
        "name": "URA Master Plan 2019 Land Use",
        "dataset_id": "d_90d86daa5bfaa371668b84fa5f01424f",
        "filename": "master_plan_land_use.geojson",
    },
    {
        "name": "HDB Existing Building",
        "dataset_id": "d_16b157c52ed637edd6ba1232e026258d",
        "filename": "hdb_existing_building.geojson",
    },
]

RADII = [500, 1000, 1500]

print(f"Base directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Radii: {RADII}m")

Base directory: /home/dmgadmin/SMU/IS630/toto-project
Data directory: /home/dmgadmin/SMU/IS630/toto-project/data
Radii: [500, 1000, 1500]m


## 2. Land Use Category Mapping

URA Master Plan 2019 has 33 zoning types (the `LU_DESC` field). Grouped them into 6 broad categories for analysis.

| Category | URA Zoning Types | Notes |
|----------|------------------|--------------------|
| **residential** | RESIDENTIAL, RESIDENTIAL / INSTITUTION, RESIDENTIAL WITH COMMERCIAL AT 1ST STOREY | Housing zones, including HDB estates and private condos |
| **commercial** | COMMERCIAL, HOTEL, BUSINESS 1/2, BUSINESS PARK | Offices, retail, hotels, industrial-commercial |
| **mixed** | COMMERCIAL & RESIDENTIAL | Integrated developments (e.g. mixed-use condo with mall) |
| **institutional** | CIVIC & COMMUNITY, EDUCATIONAL, PLACE OF WORSHIP, HEALTH | Schools, hospitals, community clubs, religious buildings |
| **open** | OPEN SPACE, PARK, SPORTS & RECREATION, BEACH | Green space, parks, sports facilities |
| **infrastructure** | ROAD, MRT/LRT, UTILITY, WATERBODY, PORT/AIRPORT | Transport and utilities (excluded from rc_ratio) |

In [2]:
LU_CATEGORY = {
    "RESIDENTIAL": "residential",
    "RESIDENTIAL / INSTITUTION": "residential",
    "RESIDENTIAL WITH COMMERCIAL AT 1ST STOREY": "residential",
    "COMMERCIAL & RESIDENTIAL": "mixed",
    "COMMERCIAL": "commercial",
    "COMMERCIAL / INSTITUTION": "commercial",
    "HOTEL": "commercial",
    "BUSINESS 1": "commercial",
    "BUSINESS 1 - WHITE": "commercial",
    "BUSINESS 2": "commercial",
    "BUSINESS 2 - WHITE": "commercial",
    "BUSINESS PARK": "commercial",
    "BUSINESS PARK - WHITE": "commercial",
    "CIVIC & COMMUNITY INSTITUTION": "institutional",
    "EDUCATIONAL INSTITUTION": "institutional",
    "PLACE OF WORSHIP": "institutional",
    "HEALTH & MEDICAL CARE": "institutional",
    "OPEN SPACE": "open",
    "PARK": "open",
    "SPORTS & RECREATION": "open",
    "BEACH AREA": "open",
    "ROAD": "infrastructure",
    "TRANSPORT FACILITIES": "infrastructure",
    "LIGHT RAPID TRANSIT": "infrastructure",
    "MASS RAPID TRANSIT": "infrastructure",
    "UTILITY": "infrastructure",
    "WATERBODY": "infrastructure",
    "PORT / AIRPORT": "infrastructure",
    "RESERVE SITE": "other",
    "SPECIAL USE": "other",
    "WHITE": "other",
    "AGRICULTURE": "other",
    "CEMETERY": "other",
}

print(f"Mapped {len(LU_CATEGORY)} URA zoning types to {len(set(LU_CATEGORY.values()))} categories")

Mapped 33 URA zoning types to 7 categories


## 3. Helper Functions

**Haversine formula:** computes distance in metres between two lat/long points.

**Polygon centroid:** averages all boundary vertices of a GeoJSON polygon to get a single representative point. Centroid error is negligible.

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    r = 6371000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return r * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def polygon_centroid(geom):
    coords = geom["coordinates"]
    pts = []
    if geom["type"] == "Polygon":
        pts = coords[0]
    elif geom["type"] == "MultiPolygon":
        for poly in coords:
            pts.extend(poly[0])
    if not pts:
        return None, None
    return sum(p[1] for p in pts) / len(pts), sum(p[0] for p in pts) / len(pts)


def download_if_missing(dataset_id, output_path):
    if output_path.exists():
        print(f"SKIP")
        return True
    print(f"Downloading {output_path.name}...")
    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    try:
        req = Request(poll_url, headers=API_HEADERS)
        with urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode())
            dl_url = data.get("data", {}).get("url")
            if dl_url:
                with urlopen(Request(dl_url, headers=API_HEADERS), timeout=600) as dl:
                    content = dl.read()
                    with open(output_path, "wb") as f:
                        f.write(content)
                    print(f"  Saved ({len(content):,} bytes)")
                    return True
    except (HTTPError, URLError) as e:
        print(f"Failed: {e}")
    print(f"https://data.gov.sg/datasets/{dataset_id}/view")
    return False

Helper functions defined.


## 4. Download Geospatial Datasets

Downloads two GeoJSON files from data.gov.sg:
- **URA Master Plan 2019 Land Use** (~166MB): 113,212 polygons, each tagged with `LU_DESC` (zoning type) and `SHAPE.AREA` (area in sq m)
- **HDB Existing Building** (~54MB): 13,404 HDB block footprints with `POSTAL_COD` and `BLK_NO`

In [4]:
for ds in GEODATASETS:
    download_if_missing(ds["dataset_id"], SUPP_DIR / ds["filename"])

  [SKIP] master_plan_land_use.geojson (174,200,577 bytes)
  [SKIP] hdb_existing_building.geojson (56,778,473 bytes)


## 5. Extract Centroids from GeoJSON

### How centroids are derived

Each GeoJSON polygon has a coordinates array of boundary vertices (lat/lon pairs). Compute the centroid by averaging all vertex latitudes and longitudes separately. This produces one representative (lat/long) point per polygon.

For **land use** GeoJSON, we also keep the polygon's SHAPE.AREA (official area in sq m from the SVY21 projection) and its mapped lu_category. Result: CSV of ~113K rows with columns lu_category, latitude, longitude, area_sqm.

For **HDB buildings** GeoJSON, we just keep the centroid coordinates. Result: ~13K rows with columns latitude, longitude.

In [ ]:
def extract_land_use_centroids():
    cache = SUPP_DIR / "land_use_centroids.csv"
    if cache.exists():
        centroids = []
        with open(cache, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                centroids.append((
                    row["lu_category"],
                    float(row["latitude"]),
                    float(row["longitude"]),
                    float(row["area_sqm"]),
                ))
        print(f"Loaded {len(centroids)} from {cache.name}")
        return centroids

    src = SUPP_DIR / "master_plan_land_use.geojson"
    print(f"Parsing {src.name}")
    with open(src, encoding="utf-8") as f:
        gj = json.load(f)

    centroids = []
    for feature in gj["features"]:
        props = feature["properties"]
        lu_desc = (props.get("LU_DESC") or "").strip()
        area = float(props.get("SHAPE.AREA", 0) or 0)
        cat = LU_CATEGORY.get(lu_desc, "other")
        geom = feature.get("geometry")
        if geom is None:
            continue
        lat, lon = polygon_centroid(geom)
        if lat is None:
            continue
        centroids.append((cat, lat, lon, area))

    with open(cache, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["lu_category", "latitude", "longitude", "area_sqm"])
        for cat, lat, lon, area in centroids:
            w.writerow([cat, lat, lon, area])

    print(f"Extracted {len(centroids)} to {cache.name}")
    del gj
    return centroids


def extract_hdb_block_centroids():
    cache = SUPP_DIR / "hdb_block_centroids.csv"
    if cache.exists():
        blocks = []
        with open(cache, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                blocks.append((float(row["latitude"]), float(row["longitude"])))
        print(f"Loaded {len(blocks)} from {cache.name}")
        return blocks

    src = SUPP_DIR / "hdb_existing_building.geojson"
    print(f"Parsing {src.name}")
    with open(src, encoding="utf-8") as f:
        gj = json.load(f)

    blocks = []
    for feature in gj["features"]:
        geom = feature.get("geometry")
        if geom is None:
            continue
        lat, lon = polygon_centroid(geom)
        if lat is None:
            continue
        blocks.append((lat, lon))

    with open(cache, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["latitude", "longitude"])
        for lat, lon in blocks:
            w.writerow([lat, lon])

    print(f"Extracted {len(blocks)} to {cache.name}")
    del gj
    return blocks


lu_centroids = extract_land_use_centroids()
hdb_blocks = extract_hdb_block_centroids()

cat_counts = defaultdict(int)
cat_area = defaultdict(float)
for cat, lat, lon, area in lu_centroids:
    cat_counts[cat] += 1
    cat_area[cat] += area

print(f"\nLand use breakdown:")
for cat in sorted(cat_counts.keys()):
    print(f"  {cat:15s}  {cat_counts[cat]:>6,} polygons  {cat_area[cat]/1e6:>8.1f} sq km")

Loaded 113212 cached land use centroids from land_use_centroids.csv
Loaded 13404 cached HDB block centroids from hdb_block_centroids.csv

Land use breakdown:
  commercial       14,553 polygons     128.8 sq km
  infrastructure    9,167 polygons     189.5 sq km
  institutional     2,077 polygons      28.3 sq km
  mixed               905 polygons       5.1 sq km
  open              2,463 polygons     148.1 sq km
  other             1,145 polygons     153.5 sq km
  residential      82,902 polygons     131.6 sq km


## 6. Load Geocoded Outlets and Planning Areas

Loads the 375 geocoded outlets from `outlets_geocoded.csv` (produced by `build_dataset.py`). Also loads the planning area boundary GeoJSON to assign regions and fill in any missing planning area values via nearest centroid lookup.

In [ ]:
pa_path = SUPP_DIR / "planning_area_boundary.geojson"
with open(pa_path, encoding="utf-8") as f:
    pa_gj = json.load(f)

pa_centroids = {}
pa_regions = {}
for feature in pa_gj["features"]:
    props = feature["properties"]
    name = props["PLN_AREA_N"]
    pa_regions[name] = props["REGION_N"]
    geom = feature.get("geometry")
    if geom:
        lat, lon = polygon_centroid(geom)
        if lat is not None:
            pa_centroids[name] = (lat, lon)

print(f"Loaded {len(pa_centroids)} planning areas across {len(set(pa_regions.values()))} regions")

input_path = DATA_DIR / "outlets_geocoded.csv"
with open(input_path, newline="", encoding="utf-8") as f:
    raw = list(csv.DictReader(f))

outlets = []
for o in raw:
    if o.get("geocode_status") != "OK":
        continue
    try:
        lat = float(o["latitude"])
        lon = float(o["longitude"])
    except (ValueError, KeyError):
        continue

    pa = (o.get("planning_area") or "").strip().upper()
    if not pa:
        best_pa, best_d = "", float("inf")
        for name, (clat, clon) in pa_centroids.items():
            d = haversine_m(lat, lon, clat, clon)
            if d < best_d:
                best_d = d
                best_pa = name
        if best_pa and best_d < 5000:
            pa = best_pa

    outlets.append({
        "outlet_name": o["outlet_name"],
        "postal_code": o.get("postal_code", ""),
        "outlet_type": o.get("outlet_type", ""),
        "group1_wins": int(o.get("group1_wins", 0)),
        "group2_wins": int(o.get("group2_wins", 0)),
        "combined_wins": int(o.get("combined_wins", 0)),
        "source": o.get("source", ""),
        "latitude": lat,
        "longitude": lon,
        "onemap_address": o.get("onemap_address", ""),
        "planning_area": pa,
        "region": pa_regions.get(pa, ""),
        "geocode_status": "OK",
    })

print(f"Loaded {len(outlets)} outlets")

Loaded 55 planning areas across 5 regions
Loaded 375 geocoded outlets


## 7. Compute Land Use Profiles

For each outlet at each radius (500m, 1000m, 1500m):

1. **Bounding box pre-filter:** discard all centroids outside a lat/long box (+/- 0.015 degrees). This reduces the inner loop from 113K to about 1-3k items per outlet.
2. **Haversine distance:** compute exact distance from outlet to each centroid.
3. **Sum areas by category:** for centroids within the radius, sum their area_sqm grouped by land use category.
4. **Count HDB blocks:** count HDB block centroids within the radius.
5. **Compute rc_ratio:** residential_area / (residential_area + commercial_area), where mixed-use area is split 50/50.

### Neighbouirhood classification (based on 1km radius)

| Type | Rule |
|------|------|
| residential | rc_ratio >= 0.65 AND hdb_blocks >= 5 |
| commercial | rc_ratio <= 0.35 |
| mixed | everything else |

In [ ]:
max_r = max(RADII)
deg_box = max_r / 111320.0 * 1.15

start = time.time()

for idx, outlet in enumerate(outlets):
    olat = outlet["latitude"]
    olon = outlet["longitude"]

    nearby_lu = [
        (cat, clat, clon, area)
        for cat, clat, clon, area in lu_centroids
        if abs(clat - olat) <= deg_box and abs(clon - olon) <= deg_box
    ]
    lu_dists = [
        (cat, area, haversine_m(olat, olon, clat, clon))
        for cat, clat, clon, area in nearby_lu
    ]

    nearby_hdb = [
        haversine_m(olat, olon, blat, blon)
        for blat, blon in hdb_blocks
        if abs(blat - olat) <= deg_box and abs(blon - olon) <= deg_box
    ]

    for radius in RADII:
        area_by_cat = defaultdict(float)
        for cat, area, dist in lu_dists:
            if dist <= radius:
                area_by_cat[cat] += area

        outlet[f"res_area_{radius}m"] = round(area_by_cat.get("residential", 0))
        outlet[f"com_area_{radius}m"] = round(area_by_cat.get("commercial", 0))
        outlet[f"mixed_area_{radius}m"] = round(area_by_cat.get("mixed", 0))
        outlet[f"inst_area_{radius}m"] = round(area_by_cat.get("institutional", 0))
        outlet[f"open_area_{radius}m"] = round(area_by_cat.get("open", 0))
        outlet[f"hdb_blocks_{radius}m"] = sum(1 for d in nearby_hdb if d <= radius)

        res_total = area_by_cat.get("residential", 0) + area_by_cat.get("mixed", 0) * 0.5
        com_total = area_by_cat.get("commercial", 0) + area_by_cat.get("mixed", 0) * 0.5
        denom = res_total + com_total
        outlet[f"rc_ratio_{radius}m"] = round(res_total / denom, 4) if denom > 0 else 0.5

    rc = outlet["rc_ratio_1000m"]
    hdb = outlet["hdb_blocks_1000m"]
    if rc >= 0.65 and hdb >= 5:
        outlet["neighborhood_type"] = "residential"
    elif rc <= 0.35:
        outlet["neighborhood_type"] = "commercial"
    else:
        outlet["neighborhood_type"] = "mixed"

    areas_1000 = {
        "residential": outlet["res_area_1000m"],
        "commercial": outlet["com_area_1000m"],
        "mixed": outlet["mixed_area_1000m"],
        "institutional": outlet["inst_area_1000m"],
        "open": outlet["open_area_1000m"],
    }
    outlet["dominant_landuse_1000m"] = max(areas_1000, key=areas_1000.get) if any(areas_1000.values()) else "unknown"
    outlet["landuse_diversity_1000m"] = sum(1 for v in areas_1000.values() if v > 0)

    if (idx + 1) % 100 == 0:
        print(f"[{idx+1}/{len(outlets)}]")

for outlet in outlets:
    hdb = outlet["hdb_blocks_1000m"]
    wins = outlet["combined_wins"]
    outlet["win_rate_hdb_1000m"] = round(wins / hdb, 6) if hdb > 0 and wins > 0 else 0.0

elapsed = time.time() - start
print(f"{elapsed:.1f}s")

  [100/375] computed...


  [200/375] computed...


  [300/375] computed...


Done in 7.2s


## Output

In [ ]:
output_path = OUT_DIR / "outlets_geodata.csv"
fieldnames = [
    "outlet_name", "postal_code", "outlet_type",
    "group1_wins", "group2_wins", "combined_wins", "source",
    "latitude", "longitude", "onemap_address", "planning_area", "region",
    "geocode_status",
    "res_area_500m", "com_area_500m", "mixed_area_500m",
    "inst_area_500m", "open_area_500m", "hdb_blocks_500m", "rc_ratio_500m",
    "res_area_1000m", "com_area_1000m", "mixed_area_1000m",
    "inst_area_1000m", "open_area_1000m", "hdb_blocks_1000m", "rc_ratio_1000m",
    "res_area_1500m", "com_area_1500m", "mixed_area_1500m",
    "inst_area_1500m", "open_area_1500m", "hdb_blocks_1500m", "rc_ratio_1500m",
    "neighborhood_type", "dominant_landuse_1000m", "landuse_diversity_1000m",
    "win_rate_hdb_1000m",
]

with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(outlets)

## How to read the output

Each row in outlets_geodata.csv represents one outlet. The land use columns:

- res_area_500m = total residential zoned area (sqm) within 500m of this outlet
- com_area_1000m = total commercial/business zoned area within 1km
- rc_ratio_1500m = residential share within 1.5km (0 = all commercial, 1 = all residential)

### Key columns for statistical analysis

| Column | Use in analysis |
|--------|----------------|
| neighbourhood_type | Grouping variable for t-tests, ANOVA, chi-squared |
| rc_ratio_1000m | Continuous predictor for regression |
| hdb_blocks_1000m | Residential density control variable |
| combined_wins | Response variable |
| win_rate_hdb_1000m | Normalised response (wins per nearby HDB block) |

In [ ]:
res_ct = sum(1 for o in outlets if o["neighbourhood_type"] == "residential")
com_ct = sum(1 for o in outlets if o["neighbourhood_type"] == "commercial")
mix_ct = sum(1 for o in outlets if o["neighbourhood_type"] == "mixed")

def avg_wins(lst):
    return sum(o["combined_wins"] for o in lst) / len(lst) if lst else 0

def med_wins(lst):
    if not lst:
        return 0
    vals = sorted(o["combined_wins"] for o in lst)
    n = len(vals)
    return vals[n // 2] if n % 2 else (vals[n // 2 - 1] + vals[n // 2]) / 2

res_out = [o for o in outlets if o["neighbourhood_type"] == "residential"]
com_out = [o for o in outlets if o["neighbourhood_type"] == "commercial"]
mix_out = [o for o in outlets if o["neighbourhood_type"] == "mixed"]

dom_counts = defaultdict(int)
for o in outlets:
    dom_counts[o["dominant_landuse_1000m"]] += 1

print(f"Total outlets:  {len(outlets)}")
print(f"Residential:    {res_ct}")
print(f"Commercial:     {com_ct}")
print(f"Mixed:          {mix_ct}")
print(f"Dominant LU:    {dict(dom_counts)}")
print(f"\nAverage combined wins by neighbourhood:")
print(f"  Residential:  {avg_wins(res_out):.1f} (median {med_wins(res_out):.0f}, n={len(res_out)})")
print(f"  Commercial:   {avg_wins(com_out):.1f} (median {med_wins(com_out):.0f}, n={len(com_out)})")
print(f"  Mixed:        {avg_wins(mix_out):.1f} (median {med_wins(mix_out):.0f}, n={len(mix_out)})")

print(f"\nTop 10 by combined wins:")
top10 = sorted(outlets, key=lambda x: x["combined_wins"], reverse=True)[:10]
for i, o in enumerate(top10, 1):
    print(f"  {i:2d}. {o['outlet_name'][:30]:<30s}  wins={o['combined_wins']:>4d}  "
          f"rc={o['rc_ratio_1000m']:.2f}  hdb={o['hdb_blocks_1000m']:>3d}  {o['neighbourhood_type']}")

Total outlets:  375
Residential:    281
Commercial:     31
Mixed:          63
Dominant LU:    {'commercial': 45, 'residential': 317, 'open': 12, 'institutional': 1}

Average combined wins by neighborhood:
  Residential:  27.0 (median 24, n=281)
  Commercial:   30.5 (median 18, n=31)
  Mixed:        26.4 (median 22, n=63)

Top 10 by combined wins:
   1. Tong Aik Huat                   wins= 166  rc=0.97  hdb=355  residential
   2. Delisia Agency Pte Ltd          wins= 126  rc=0.33  hdb= 69  commercial
   3. NTUC FairPrice NEX              wins= 100  rc=0.96  hdb=171  residential
   4. Singapore Pools People's Park   wins=  96  rc=0.30  hdb= 45  commercial
   5. Ng Teo Guan Self Service        wins=  94  rc=0.43  hdb=149  mixed
   6. Tan Wee Fong Trading            wins=  88  rc=0.72  hdb=193  residential
   7. Singapore Pools Chinatown Poin  wins=  82  rc=0.27  hdb= 43  commercial
   8. NTUC FairPrice Tampines Mall    wins=  78  rc=0.88  hdb=310  residential
   9. Singapore Pools Clemen

In [10]:
n = len(outlets)
wins = [o["combined_wins"] for o in outlets]
mean_w = sum(wins) / n

print("Pearson r: combined_wins vs ...")
for col in ["hdb_blocks_1000m", "res_area_1000m", "com_area_1000m", "rc_ratio_1000m",
            "hdb_blocks_500m", "res_area_500m", "com_area_500m", "rc_ratio_500m"]:
    vals = [o[col] for o in outlets]
    mean_v = sum(vals) / n
    cov = sum((w - mean_w) * (v - mean_v) for w, v in zip(wins, vals)) / n
    std_w = (sum((w - mean_w) ** 2 for w in wins) / n) ** 0.5
    std_v = (sum((v - mean_v) ** 2 for v in vals) / n) ** 0.5
    r = cov / (std_w * std_v) if std_w > 0 and std_v > 0 else 0
    print(f"  {col:25s}  r = {r:+.4f}")

Pearson r: combined_wins vs ...
  hdb_blocks_1000m           r = +0.1134
  res_area_1000m             r = +0.0471
  com_area_1000m             r = +0.0162
  rc_ratio_1000m             r = -0.0164
  hdb_blocks_500m            r = +0.1383
  res_area_500m              r = +0.0784
  com_area_500m              r = -0.0410
  rc_ratio_500m              r = +0.0215
